In [ ]:
#Install libraries
!pip install pulp -q

# Planificación de inventarios multiproducto y multi-período en farmacia

## Contexto real
Este problema se basa en la gestión de inventarios en una farmacia ubicada en Barranquilla, donde se deben tomar decisiones de abastecimiento a lo largo de varios períodos. El objetivo es garantizar la disponibilidad de medicamentos mientras se minimizan los costos asociados al inventario y a los pedidos. Este tipo de modelos es ampliamente utilizado en cadenas de suministro farmacéuticas, donde existen restricciones de almacenamiento, demanda variable y condiciones operativas específicas

## Datos del problema

### Productos:
Medicamentos tipo A (ej. antibióticos 1)  
Medicamentos tipo B (ej. antibióticos 2)

### Períodos:
t=1,2,3

### Costos de compra (por unidad):
Producto A: 10, 12, 11  
Producto B: 8, 9, 10

### Costos de almacenamiento:
Producto A: 2 por unidad  
Producto B: 3 por unidad

### Demandas:
Producto A: 30, 40, 35  
Producto B: 20, 25, 30

### Capacidad:
Capacidad máxima de almacenamiento: 100 unidades por período

### Inventario inicial:
0 unidades para todos

## Supuestos
La demanda debe satisfacerse completamente en cada período.  
Los productos pueden almacenarse para períodos futuros.  
No hay pérdidas de inventario.  
Los costos son lineales.  
Los pedidos pueden realizarse en cada período.

## Nivel de complejidad
Este problema es de nivel alto, ya que integra decisiones interdependientes en múltiples períodos y múltiples productos dentro de un entorno real de cadena de suministro. Su estructura multi-período implica dependencias temporales entre decisiones de inventario y abastecimiento, lo que incrementa significativamente la complejidad del modelo y su resolución.

In [ ]:
# ============================================================
# PROGRAMACIÓN LINEAL ENTERA MIXTA
# Planificación de inventarios multiproducto y multi-período
# Farmacia en Barranquilla
# Ejercicio #5 - avanzado
# ============================================================

from pulp import LpMinimize, LpProblem, LpVariable, LpStatus, value, LpBinary

# ============================================================
# 1) DATOS
# ============================================================

P = ["A", "B"]
T = ["1", "2", "3"]

c_compra = {
    ("A", "1"): 10, ("A", "2"): 12, ("A", "3"): 11,
    ("B", "1"): 8,  ("B", "2"): 9,  ("B", "3"): 10
}

c_inv = {
    "A": 2,
    "B": 3
}

demanda = {
    ("A", "1"): 30, ("A", "2"): 40, ("A", "3"): 35,
    ("B", "1"): 20, ("B", "2"): 25, ("B", "3"): 30
}

inventario_inicial = {
    "A": 0,
    "B": 0
}

capacidad_almacenamiento = 100
M = 200

# ============================================================
# 2) FUNCIONES PARA GENERAR SALIDA PROGRAMADA
# ============================================================

def mostrar(titulo, lineas):
    print(titulo)
    for linea in lineas:
        print(linea)
    print()

def sumatoria(indices, expresion):
    return "".join(f"Σ{indice}" for indice in indices) + f" {expresion}"

def generar_variables():
    variables = {
        "qpt": "cantidad comprada del producto p en el período t",
        "ipt": "inventario del producto p al final del período t",
        "ypt": "1 si se realiza pedido del producto p en el período t"
    }

    return [
        f"{variable} = {descripcion}"
        for variable, descripcion in variables.items()
    ]

def generar_funcion_objetivo():
    return [
        "Min Z = "
        + sumatoria(["p", "t"], "cpt·qpt")
        + " + "
        + sumatoria(["p", "t"], "hp·ipt")
    ]

def generar_restricciones():
    restricciones = {
        "Balance de inventario": [
            "i[p,t-1] + q[p,t] - d[p,t] = i[p,t]"
        ],
        "Capacidad de almacenamiento": [
            sumatoria(["p"], "i[p,t]") + f" <= {capacidad_almacenamiento}"
        ],
        "Activación de pedido": [
            "q[p,t] <= M·y[p,t]"
        ],
        "No negatividad y binarias": [
            "q[p,t] >= 0",
            "i[p,t] >= 0",
            "y[p,t] = 0 o 1"
        ]
    }

    lineas = []

    for titulo, contenido in restricciones.items():
        lineas.append(f"{titulo}:")
        lineas.extend(contenido)
        lineas.append("")

    return lineas

# ============================================================
# 3) MODELO
# ============================================================

modelo = LpProblem("Inventarios_Farmacia", LpMinimize)

q = {
    (p, t): LpVariable(f"q{p}{t}", lowBound=0)
    for p in P
    for t in T
}

i = {
    (p, t): LpVariable(f"i{p}{t}", lowBound=0)
    for p in P
    for t in T
}

y = {
    (p, t): LpVariable(f"y{p}{t}", cat=LpBinary)
    for p in P
    for t in T
}

# ============================================================
# 4) FUNCIÓN OBJETIVO CON SUMATORIAS REALES
# ============================================================

modelo += (
    sum(c_compra[p, t] * q[p, t] for p in P for t in T)
    +
    sum(c_inv[p] * i[p, t] for p in P for t in T)
), "Costo_Total"

# ============================================================
# 5) RESTRICCIONES CON SUMATORIAS REALES
# ============================================================

# Balance de inventario
for p in P:
    for t in T:
        if t == "1":
            modelo += (
                inventario_inicial[p] + q[p, t] - demanda[p, t] == i[p, t]
            ), f"Balance_{p}_t{t}"
        else:
            t_anterior = str(int(t) - 1)
            modelo += (
                i[p, t_anterior] + q[p, t] - demanda[p, t] == i[p, t]
            ), f"Balance_{p}_t{t}"

# Capacidad de almacenamiento
for t in T:
    modelo += (
        sum(i[p, t] for p in P) <= capacidad_almacenamiento
    ), f"Capacidad_t{t}"

# Activación de pedido
for p in P:
    for t in T:
        modelo += (
            q[p, t] <= M * y[p, t]
        ), f"Activacion_{p}_t{t}"

# ============================================================
# 6) SOLUCIÓN
# ============================================================

modelo.solve()
estado = LpStatus[modelo.status]

# ============================================================
# 7) OUTPUT PROGRAMADO
# ============================================================

mostrar("1) VARIABLES DE DECISIÓN", generar_variables())

mostrar("2) FUNCIÓN OBJETIVO", generar_funcion_objetivo())

mostrar("3) RESTRICCIONES", generar_restricciones())

print("4) SOLUCIÓN")
print("Estado:", estado)
print()

if estado == "Optimal":

    print("Compras óptimas:")
    for p in P:
        for t in T:
            print(f"q{p}{t} = {q[p, t].varValue}")
    print()

    print("Inventarios óptimos:")
    for p in P:
        for t in T:
            print(f"i{p}{t} = {i[p, t].varValue}")
    print()

    print("Activación de pedidos:")
    for p in P:
        for t in T:
            print(f"y{p}{t} = {y[p, t].varValue}")
    print()

    print("Costo mínimo total:")
    print("Z =", value(modelo.objective))
    print()

    print("Verificación de capacidad de almacenamiento:")
    for t in T:
        uso = sum(i[p, t].varValue for p in P)
        print(f"t{t} = {uso} de {capacidad_almacenamiento}")

else:
    print("El modelo no tiene solución óptima.")

1) VARIABLES DE DECISIÓN
qpt = cantidad comprada del producto p en el período t
ipt = inventario del producto p al final del período t
ypt = 1 si se realiza pedido del producto p en el período t

2) FUNCIÓN OBJETIVO
Min Z = ΣpΣt cpt·qpt + ΣpΣt hp·ipt

3) RESTRICCIONES
Balance de inventario:
i[p,t-1] + q[p,t] - d[p,t] = i[p,t]

Capacidad de almacenamiento:
Σp i[p,t] <= 100

Activación de pedido:
q[p,t] <= M·y[p,t]

No negatividad y binarias:
q[p,t] >= 0
i[p,t] >= 0
y[p,t] = 0 o 1


4) SOLUCIÓN
Estado: Optimal

Compras óptimas:
qA1 = 70.0
qA2 = 0.0
qA3 = 35.0
qB1 = 20.0
qB2 = 25.0
qB3 = 30.0

Inventarios óptimos:
iA1 = 40.0
iA2 = 0.0
iA3 = 0.0
iB1 = 0.0
iB2 = 0.0
iB3 = 0.0

Activación de pedidos:
yA1 = 1.0
yA2 = 1.0
yA3 = 1.0
yB1 = 1.0
yB2 = 1.0
yB3 = 1.0

Costo mínimo total:
Z = 1850.0

Verificación de capacidad de almacenamiento:
t1 = 40.0 de 100
t2 = 0.0 de 100
t3 = 0.0 de 100
